In [0]:
import requests, json

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

metastore_id = "97d13058-1c7f-4a10-800c-ab6ab6afbc16"

# PATCH metastore to enable public DBFS root access
response = requests.patch(
    f"{host}/api/2.1/unity-catalog/metastores/{metastore_id}",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"enable_public_dbfs": True}
)
print(f"PATCH Status: {response.status_code}")
if response.text:
    print(json.dumps(response.json(), indent=2))
else:
    print("Success - no content returned")

# Verify it was set
response2 = requests.get(
    f"{host}/api/2.1/unity-catalog/metastores/{metastore_id}",
    headers={"Authorization": f"Bearer {token}"}
)
data = response2.json()
print(f"\nenable_public_dbfs is now: {data.get('enable_public_dbfs', 'not in response - check if 200 means enabled')}")
print(f"Verification GET status: {response2.status_code}")

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@marufazureproject.dfs.core.windows.net/DimUser")

In [0]:
# Reset all CDC watermarks to early date
bronze_base = "abfss://bronze@marufazureproject.dfs.core.windows.net"
cdc_folders = ["DimArtist_cdc", "DimDate_cdc", "DimTrack_cdc", "DimUser_cdc", "FactStream_cdc"]
early_date = '{"cdc":"2000-01-01T00:00:00"}'
for folder in cdc_folders:
    path = f"{bronze_base}/{folder}/cdc.json"
    try:
        dbutils.fs.put(path, early_date, overwrite=True)
        print(f"Reset {folder}/cdc.json to {early_date}")
    except Exception as e:
        print(f"Error resetting {folder}: {e}")

In [0]:
df2 = spark.read.format("parquet").load("abfss://bronze@marufazureproject.dfs.core.windows.net/DimUser")
display(df2)

AUTO LOADER

In [0]:

for q in spark.streams.active: q.stop()
bronze_base = "abfss://bronze@marufazureproject.dfs.core.windows.net"
for folder in ["DimArtist", "DimDate", "DimTrack", "FactStream"]:
    try: dbutils.fs.rm(f"{bronze_base}/{folder}", recurse=True)
    except: pass

silver_base = "abfss://silver@marufazureproject.dfs.core.windows.net"
for folder in ["DimArtist", "DimDate", "DimTrack", "FactStream"]:
    try: dbutils.fs.rm(f"{silver_base}/{folder}/checkpoint", recurse=True)
    except: pass
print("Silver checkpoints cleared")
print("Cleanup done")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from datetime import date, datetime, timedelta
import random
bronze_base = "abfss://bronze@marufazureproject.dfs.core.windows.net"
genres=["Pop","Rock","Hip-Hop","Jazz","Classical","Electronic"]
countries=["USA","UK","Canada","Australia","Germany","Brazil"]
artist_rows=[Row(artist_id=i,artist_name=f"Artist_{i}",genre=genres[i%6],country=countries[i%6],updated_at=datetime(2024,1,1)) for i in range(1,501)]
df_a=spark.createDataFrame(artist_rows,schema=StructType([StructField("artist_id",IntegerType(),False),StructField("artist_name",StringType(),False),StructField("genre",StringType(),True),StructField("country",StringType(),True),StructField("updated_at",TimestampType(),True)]))
df_a.write.mode("overwrite").parquet(f"{bronze_base}/DimArtist")
print(f"DimArtist: {df_a.count()} rows")
start=date(2024,1,1)
date_rows=[Row(date_key=int((start+timedelta(d)).strftime("%Y%m%d")),date=(start+timedelta(d)),day=(start+timedelta(d)).day,month=(start+timedelta(d)).month,year=2024,weekday=(start+timedelta(d)).strftime("%A")) for d in range(365)]
df_d=spark.createDataFrame(date_rows,schema=StructType([StructField("date_key",IntegerType(),False),StructField("date",DateType(),False),StructField("day",IntegerType(),True),StructField("month",IntegerType(),True),StructField("year",IntegerType(),True),StructField("weekday",StringType(),True)]))
df_d.write.mode("overwrite").parquet(f"{bronze_base}/DimDate")
print(f"DimDate: {df_d.count()} rows")
track_rows=[Row(track_id=i,track_name=f"Track_{i}",artist_id=(i%500)+1,album_name=f"Album_{(i%200)+1}",duration_sec=180+(i%180),release_date=date(2018+(i%7),1+(i%12),1+(i%28)),updated_at=datetime(2024,1,1)) for i in range(1,1001)]
df_t=spark.createDataFrame(track_rows,schema=StructType([StructField("track_id",IntegerType(),False),StructField("track_name",StringType(),False),StructField("artist_id",IntegerType(),True),StructField("album_name",StringType(),True),StructField("duration_sec",IntegerType(),True),StructField("release_date",DateType(),True),StructField("updated_at",TimestampType(),True)]))
df_t.write.mode("overwrite").parquet(f"{bronze_base}/DimTrack")
print(f"DimTrack: {df_t.count()} rows")
stream_rows=[Row(stream_id=i,user_id=(i%500)+1,track_id=(i%1000)+1,date_key=int((date(2024,1,1)+timedelta(i%365)).strftime("%Y%m%d")),listen_duration=60+(i%300),device_type=["mobile","desktop","tablet"][i%3],stream_timestamp=datetime(2024,1+(i%12),1+(i%28),i%24,(i*7)%60)) for i in range(1,10001)]
df_s=spark.createDataFrame(stream_rows,schema=StructType([StructField("stream_id",LongType(),False),StructField("user_id",IntegerType(),True),StructField("track_id",IntegerType(),True),StructField("date_key",IntegerType(),True),StructField("listen_duration",IntegerType(),True),StructField("device_type",StringType(),True),StructField("stream_timestamp",TimestampType(),True)]))
df_s.write.mode("overwrite").parquet(f"{bronze_base}/FactStream")
print(f"FactStream: {df_s.count()} rows")
for q in spark.streams.active: q.stop()
print("Done!")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

user_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("user_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("subscription_type", StringType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("updated_at", TimestampType(), True)
])

df_user = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimUser/checkpoint")
    .schema(user_schema)
    .load("abfss://bronze@marufazureproject.dfs.core.windows.net/DimUser")
)

In [0]:
# Display df_user as a batch read from bronze (clean preview)
df_user_preview = spark.read.format("parquet").load("abfss://bronze@marufazureproject.dfs.core.windows.net/DimUser")
display(df_user_preview)

In [0]:
%sql
SELECT current_user()

In [0]:
%sql
GRANT READ FILES ON EXTERNAL LOCATION `silver` TO `mdmarufuzzaman.cs_gmail.com#ext#@mdmarufuzzamancsgmail.onmicrosoft.com`

In [0]:
%sql
GRANT WRITE FILES ON EXTERNAL LOCATION `silver` TO `mdmarufuzzaman.cs_gmail.com#ext#@mdmarufuzzamancsgmail.onmicrosoft.com`

In [0]:
# Write df_user stream to silver Delta table
query = (
    df_user
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimUser/checkpoint")
    .trigger(availableNow=True)
    .start("abfss://silver@marufazureproject.dfs.core.windows.net/DimUser")
)
query.awaitTermination()

# Display the silver table
df_silver_user = spark.read.format("delta").load("abfss://silver@marufazureproject.dfs.core.windows.net/DimUser")
display(df_silver_user)

In [0]:
print(df_user.isStreaming)

In [0]:
(df_user.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@marufazureproject.dfs.core.windows.net/DimUser/checkpoint_v2")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("spotify_catalog.silver.dim_users")
).awaitTermination()

df_silver_user = spark.read.table("spotify_catalog.silver.dim_users")
display(df_silver_user)

In [0]:
# dim_users streaming write done in cell 15 above

In [0]:
# placeholder - df_user defined above

In [0]:
# Create a managed volume for streaming checkpoints
spark.sql("CREATE VOLUME IF NOT EXISTS main.default.checkpoints")

In [0]:
# dim_users already written to silver.dim_users in cell 15

In [0]:
# Inspect schemas for remaining bronze dimensions
bronze_base = "abfss://bronze@marufazureproject.dfs.core.windows.net"
for name in ["DimArtist", "DimDate", "DimTrack", "FactStream"]:
    df_tmp = spark.read.format("parquet").load(f"{bronze_base}/{name}")
    print(f"=== {name} ===")
    df_tmp.printSchema()

In [0]:
# — DimArtist
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

artist_schema = StructType([
    StructField("artist_id",   IntegerType(), True),
    StructField("artist_name", StringType(),  True),
    StructField("genre",       StringType(),  True),
    StructField("country",     StringType(),  True),
    StructField("updated_at",  TimestampType(), True),
])

bronze_base = "abfss://bronze@marufazureproject.dfs.core.windows.net"
silver_base = "abfss://silver@marufazureproject.dfs.core.windows.net"

df_artist = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver_base}/DimArtist/checkpoint")
    .schema(artist_schema)
    .load(f"{bronze_base}/DimArtist")
)

query_artist = (
    df_artist.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver_base}/DimArtist/checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("spotify_catalog.silver.dim_artists")
)
query_artist.awaitTermination()

df_silver_artist = spark.read.table("spotify_catalog.silver.dim_artists")
display(df_silver_artist)

In [0]:
# — DimDate
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

date_schema = StructType([
    StructField("date_key", IntegerType(), True),
    StructField("date",     DateType(),    True),
    StructField("day",      IntegerType(), True),
    StructField("month",    IntegerType(), True),
    StructField("year",     IntegerType(), True),
    StructField("weekday",  StringType(),  True),
])

bronze_base = "abfss://bronze@marufazureproject.dfs.core.windows.net"
silver_base = "abfss://silver@marufazureproject.dfs.core.windows.net"

df_date = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver_base}/DimDate/checkpoint")
    .schema(date_schema)
    .load(f"{bronze_base}/DimDate")
)

query_date = (
    df_date.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver_base}/DimDate/checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("spotify_catalog.silver.dim_date")
)
query_date.awaitTermination()

df_silver_date = spark.read.table("spotify_catalog.silver.dim_date")
display(df_silver_date)

In [0]:
# — DimTrack
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

track_schema = StructType([
    StructField("track_id",     IntegerType(), True),
    StructField("track_name",   StringType(),  True),
    StructField("artist_id",    IntegerType(), True),
    StructField("album_name",   StringType(),  True),
    StructField("duration_sec", IntegerType(), True),
    StructField("release_date", DateType(),    True),
    StructField("updated_at",   TimestampType(), True),
])

bronze_base = "abfss://bronze@marufazureproject.dfs.core.windows.net"
silver_base = "abfss://silver@marufazureproject.dfs.core.windows.net"

df_track = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver_base}/DimTrack/checkpoint")
    .schema(track_schema)
    .load(f"{bronze_base}/DimTrack")
)

query_track = (
    df_track.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver_base}/DimTrack/checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("spotify_catalog.silver.dim_tracks")
)
query_track.awaitTermination()

df_silver_track = spark.read.table("spotify_catalog.silver.dim_tracks")
display(df_silver_track)

In [0]:
# — FactStream
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, StringType, TimestampType

stream_schema = StructType([
    StructField("stream_id",        LongType(),      True),
    StructField("user_id",          IntegerType(),   True),
    StructField("track_id",         IntegerType(),   True),
    StructField("date_key",         IntegerType(),   True),
    StructField("listen_duration",  IntegerType(),   True),
    StructField("device_type",      StringType(),    True),
    StructField("stream_timestamp", TimestampType(), True),
])

bronze_base = "abfss://bronze@marufazureproject.dfs.core.windows.net"
silver_base = "abfss://silver@marufazureproject.dfs.core.windows.net"

df_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver_base}/FactStream/checkpoint")
    .schema(stream_schema)
    .load(f"{bronze_base}/FactStream")
)

query_stream = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver_base}/FactStream/checkpoint")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("spotify_catalog.silver.fact_streams")
)
query_stream.awaitTermination()

df_silver_stream = spark.read.table("spotify_catalog.silver.fact_streams")
display(df_silver_stream)